
# Hierarchical PSD Inference

Sets up a small population of mock galaxies sharing the same burstiness
PSD parameters (sigma, tau). Runs PopulationFitter with VI to recover
the shared PSD hyperparameters from per-galaxy photometry, demonstrating
hierarchical Bayesian shrinkage on the intrinsic SFH scatter.

Reference: Tegmark et al. 1998 (hierarchical inference framework);
Leja et al. 2019 (rapid field inference with correlated priors).


In [ ]:
import time
import warnings

import jax
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()
obs = tengri.Observation(
    photometry=tengri.Photometry.from_names(["sdss_u", "sdss_g", "sdss_r", "sdss_i", "sdss_z"])
)

TRUE_SIGMA = 2.0
TRUE_TAU = 20.0
N_GAL = 4


def model_factory(psd_sigma=1.0, psd_tau_myr=50.0):
    """Create a SEDModel with fixed PSD."""
    return tengri.SEDModel.build(
        ssp, observation=obs,
        sfh={
            "type": "field",
            "*": tengri.FREE,
            "psd_sigma": tengri.Fixed(psd_sigma),
            "psd_tau_myr": tengri.Fixed(psd_tau_myr),
        },
        met={"type": "fixed"},
        dust={"type": "two_component", "*": tengri.FIXED},
        redshift=tengri.Fixed(0.1),
        n_grid=128,
    )


key = jax.random.PRNGKey(42)
model_gen = model_factory(psd_sigma=TRUE_SIGMA, psd_tau_myr=TRUE_TAU)
galaxies = []
for i in range(N_GAL):
    k = jax.random.fold_in(key, i)
    params = model_gen.spec.sample(k)
    mock = model_gen.mock(params, snr=20.0, key=jax.random.fold_in(k, 1))
    galaxies.append({"flux_obs": mock.flux_obs, "noise": mock.noise})
print(f"Generated {N_GAL} mock galaxies with sigma={TRUE_SIGMA}, tau={TRUE_TAU} Myr")

hfitter = tengri.PopulationFitter(
    model_factory,
    galaxies,
    psd_sigma_prior=(0.1, 4.0),
    psd_tau_prior=(1.0, 300.0),
)

t0 = time.perf_counter()
result = hfitter.run(
    "vi_linear",
    n_iterations=20,
    n_samples=4,
    n_posterior_samples=500,
    verbose=False,
    key=jax.random.PRNGKey(0),
)
elapsed = time.perf_counter() - t0
print(f"Hierarchical fit: {elapsed:.1f}s")

keys = list(result.shared_samples.keys())
sig_key = next(
    k for k in keys if "psd" in k and ("sigma" in k or "_u" in k or "amp" in k)
)
tau_key = next(k for k in keys if "psd" in k and ("tau" in k))
sig_samples = np.array(result.shared_samples[sig_key])
tau_samples = np.array(result.shared_samples[tau_key])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.hist(sig_samples, bins=30, density=True, alpha=0.7, color="steelblue")
ax1.axvline(TRUE_SIGMA, color="crimson", ls="--", lw=2, label=f"Truth = {TRUE_SIGMA}")
ax1.set_xlabel(r"$\sigma_{\rm PS}$")
ax1.set_ylabel("Density")
ax1.legend(frameon=False)

ax2.hist(tau_samples, bins=30, density=True, alpha=0.7, color="steelblue")
ax2.axvline(TRUE_TAU, color="crimson", ls="--", lw=2, label=f"Truth = {TRUE_TAU} Myr")
ax2.set_xlabel(r"$\tau_{\rm PS}$ [Myr]")
ax2.set_ylabel("Density")
ax2.legend(frameon=False)

fig.suptitle(f"Hierarchical PSD recovery ({N_GAL} galaxies, {elapsed:.0f}s)")
fig.tight_layout()
fig.savefig("plot_hierarchical.png", dpi=150, bbox_inches="tight")